High dimensional separable convolutions in natural domain

One input channel and one output channels (SISO) 

---

Kishore Kumar Tarafdar, 06-06-2025

In [1]:
pwd

'/data1/kishoretarafdar/src.port/NSLI.v00'

In [2]:
!python --version

Python 3.12.7


GPU availability?

In [3]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

2025-06-06 01:42:07.550373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749154327.573181 3080753 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749154327.580244 3080753 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-06 01:42:07.604417: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3


3

Select one GPU

        Restrict code to use a particular GPU...

In [4]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [6]:
# select_gpu = gpus[gpu_id]
memory_limit = 48#GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

3 Physical GPUs available 
Selected 1 Logical GPU with 48 GB memory limit


I0000 00:00:1749154336.629758 3080753 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49152 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


# Separable convolution strategy
        
        separable and nonseparable (referencd) 2d convolution
        outer product of kernel
        Perfect match with one channel input
        !! does not match when multiple channel input

In [ ]:
import tensorflow as tf

def separable_2d_conv(x, kernel1d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, C = x.shape

    # Step 1: Convolve over (N1)
    ## x: B, N1, N2, C 
    x1 = tf.reshape(x, [B * N2, N1, C])
    y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
    y1 = tf.reshape(y1, [B, N1, N2, C])
    y1 = tf.transpose(y1, perm=[0,2,1,3])
    ## B, N2, N1, C 
   
    # Step 2: Convolve over (N2)
    ## B, N2, N1, C
    x2 = tf.reshape(y1, [B * N1, N2, C])
    y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
    y2 = tf.reshape(y2, [B, N1, N2, C])
    y2 = tf.transpose(y2, perm=[0,2,1,3])
    ## B, N1, N2, C

    return y2

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input shape
B, N1, N2, C = 1, 8, 8, 1 #OK
# B, N1, N2, C = 1, 2, 2, 2 # 
# B, N1, N2, C = 50, 8, 8, 1 #OK
# B, N1, N2, C = 2, 2, 2, 2 #issue with multiple channels
# B, N1, N2, C = 5, 2, 3, 2
# B, N1, N2, C = 5, 8, 8, 2

# O = 2
x = tf.random.normal((B, N1, N2, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH = 3
kH = 2
kernel1d = tf.random.normal((kH, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_2d_conv(x, kernel1d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel1d.shape)
print("Output shape:", y.shape)



Input shape : (1, 8, 8, 1)
Kernel 2D shape: (2, 1, 1)
Output shape: (1, 8, 8, 1)


In [ ]:
# def foo(x):
#     C = 1
#     tf.expand_dims(tf.unstack(x, axis=-1))

[<tf.Tensor: shape=(1, 2, 2), dtype=float32, numpy=
 array([[[ 0.96406895,  1.7949798 ],
         [-0.18434155, -0.16262998]]], dtype=float32)>,
 <tf.Tensor: shape=(1, 2, 2), dtype=float32, numpy=
 array([[[ 0.25728133, -0.1631882 ],
         [ 0.96027887,  0.08398907]]], dtype=float32)>]

In [318]:
kernel2d = tf.einsum('ico,kco->ikco', kernel1d, kernel1d)
kernel2d.shape

TensorShape([2, 2, 1, 1])

In [319]:
ydef = tf.nn.convolution(x, kernel2d, padding='SAME')
ydef.shape

TensorShape([1, 8, 8, 1])

In [320]:
# Compare
diff = tf.reduce_max(tf.abs(y - ydef))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - ydef) < 1e-4).numpy())


Max absolute difference: 4.7683716e-07
Outputs match: True


np problem with multiple batches

!!issue with more than 1 channels

In [196]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [203]:
import numpy as np
np.squeeze(kernel2d)#[::-1]

array([[[[ 0.8992288 ,  0.03782616],
         [ 0.02944931,  0.5439975 ]],

        [[-0.38017905,  0.05966839],
         [ 0.12532397,  0.58402634]]],


       [[[-0.38017905,  0.05966839],
         [ 0.12532397,  0.58402634]],

        [[ 0.1607334 ,  0.09412312],
         [ 0.5333265 ,  0.6270005 ]]]], dtype=float32)

In [119]:
np.squeeze(x)

array([[-1.3135974 ,  1.6594728 ],
       [-0.42444474, -0.9835947 ]], dtype=float32)

In [126]:
np.squeeze(kernel1d)

array([-2.7352364 ,  0.54115134], dtype=float32)

In [ ]:
np.squeeze(y)

array([[-14.74033  ,  12.415377 ],
       [ -0.2637028,  -7.358782 ]], dtype=float32)

    verifying ydef

In [121]:
np.squeeze(ydef)

array([[-11.943804,  13.871271],
       [ -1.719597,  -7.358782]], dtype=float32)

In [122]:
tf.einsum('ij,ij->', np.squeeze(x), np.squeeze(kernel2d)).numpy()

np.float32(-11.943804)

In [123]:
7.4815183*1.6594728 +-1.4801768*-0.9835947

13.8712701770952

In [124]:
7.4815183*-0.42444474 + -1.4801768*-0.9835947

-1.7195970341057818

In [125]:
-0.9835947 * 7.4815183

-7.35878174783301

    verified separable 2d and non separable 2D convolutions above for batched SISO system

np.float32(-11.943804)

In [127]:
import numpy as np
np.squeeze(kernel2d)#[::-1]

array([[ 7.4815183 , -1.4801768 ],
       [-1.4801768 ,  0.29284477]], dtype=float32)

In [128]:
np.squeeze(x)

array([[-1.3135974 ,  1.6594728 ],
       [-0.42444474, -0.9835947 ]], dtype=float32)

In [129]:
np.squeeze(kernel1d)

array([-2.7352364 ,  0.54115134], dtype=float32)

In [130]:
x1 = tf.reshape(x, [B * N2, N1, C])
x1.shape

TensorShape([2, 2, 1])

In [145]:
y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
y1.shape
y1 = tf.reshape(y1, [B, N1, N2, C])
tf.squeeze(y1)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[ 4.4910254 , -4.5390506 ],
       [ 0.62868315,  2.6903641 ]], dtype=float32)>

        CHECK

In [146]:
-2.7352364*-1.3135974 + 0.54115134*1.6594728

4.4910253528389115

In [147]:
-2.7352364*1.6594728

-4.53905040736992

In [148]:
-2.7352364*-0.42444474 + 0.54115134*-0.9835947

0.628683112714638

In [149]:
-2.7352364*-0.9835947

2.69036402628708

In [155]:
y1 = tf.transpose(y1, perm=[0,2,1,3])
x2 = tf.reshape(y1, [B * N1, N2, C])
tf.squeeze(x2)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[ 4.4910254 ,  0.62868315],
       [-4.5390506 ,  2.6903641 ]], dtype=float32)>

In [156]:
x2[0,...]

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[4.4910254 ],
       [0.62868315]], dtype=float32)>

In [159]:
y2 = tf.nn.convolution(x2, kernel1d, padding='SAME')
y2 = tf.reshape(y2, [B, N1, N2, C])
y2 = tf.transpose(y2, perm=[0,2,1,3])
tf.squeeze(y2)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[-11.943804,  13.871271],
       [ -1.719597,  -7.358782]], dtype=float32)>

In [154]:
-2.7352364*4.4910254 +  0.54115134*0.62868315

-11.943803418346638

        issue with channels!!

In [228]:
kernel2d[:,:,0,0], kernel2d[:,:,1,0],

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.04413895, -0.24178982],
        [-0.24178982,  1.3245063 ]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[0.09562411, 0.0405251 ],
        [0.0405251 , 0.01717436]], dtype=float32)>)

In [229]:
kernel2d[:,:,0,1], kernel2d[:,:,1,1]

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.36423975, -0.15391304],
        [-0.15391304,  0.06503744]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[0.8157267 , 0.63113356],
        [0.63113356, 0.48831257]], dtype=float32)>)

In [ ]:
x[0,:,:,0], x[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[-0.50684583, -1.3502959 ],
        [ 0.33504048,  0.5373964 ]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.1334256 , -0.25891632],
        [ 0.12293333,  0.20609343]], dtype=float32)>)

In [251]:
kernel1d[:,0,0], kernel1d[:,1,0]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 0.21009271, -1.150872  ], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.3092315, -0.131051 ], dtype=float32)>)

In [252]:
kernel1d[:,0,1], kernel1d[:,1,1]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.6035228,  0.2550244], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.9031759 , -0.69879365], dtype=float32)>)

In [232]:
y[0,:,:,0], y[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.2576822 , -0.35678944],
        [-0.0297929 ,  0.16818404]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.1807254 , -0.45509022],
        [ 0.65922415,  0.4313671 ]], dtype=float32)>)

In [233]:
ydef[0,:,:,0], ydef[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.0413033 , -0.2059443 ],
        [-0.09504129,  0.04342761]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 0.9459787 , -0.65567625],
        [ 0.26967523,  0.36385703]], dtype=float32)>)

In [244]:
##
# 0.04413895*-0.50684583 + -0.24178982*-1.3502959 + -0.24178982*0.33504048 + 1.3245063*0.5373964 + 
# 0.09562411*1.1334256 + 0.0405251*-0.25891632 + 0.0405251*0.12293333 + 0.01717436*0.20609343  

In [257]:
tf.einsum('ij,ij->', np.squeeze(kernel2d[:,:,0,0]), np.squeeze(x[0,:,:,0])).numpy() + tf.einsum('ij,ij->', np.squeeze(kernel2d[:,:,1,0]), np.squeeze(x[0,:,:,1])).numpy()

np.float32(1.0413033)

---

In [248]:
x1 = tf.reshape(x, [B * N2, N1, C])
y1 = tf.nn.convolution(x1, kernel1d, padding='SAME')
y1 = tf.reshape(y1, [B, N1, N2, C])
y1.shape

TensorShape([1, 2, 2, 2])

In [249]:
y1[0,:,:,0], y1[0,:,:,1] 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.1309735 , -0.20362225],
        [-0.61310846,  0.04917248]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[-0.881219  ,  1.0487813 ],
        [-0.32020256, -0.51046956]], dtype=float32)>)

In [253]:
kernel1d[:,0,0], kernel1d[:,1,0]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 0.21009271, -1.150872  ], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.3092315, -0.131051 ], dtype=float32)>)

In [254]:
kernel1d[:,0,1], kernel1d[:,1,1]

(<tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.6035228,  0.2550244], dtype=float32)>,
 <tf.Tensor: shape=(2,), dtype=float32, numpy=array([-0.9031759 , -0.69879365], dtype=float32)>)

In [255]:
x[0,:,:,0], x[0,:,:,1], 

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[-0.50684583, -1.3502959 ],
        [ 0.33504048,  0.5373964 ]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[ 1.1334256 , -0.25891632],
        [ 0.12293333,  0.20609343]], dtype=float32)>)

In [294]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)